# Program 2: Feed-forward Neural Network for IMDB Movie Review Classification


### Step 1: Libraries & Device Setup
* **`torch`**: The main PyTorch library.
* **`nn`**: Neural network modules (layers, loss functions).
* **`optim`**: Optimizers (like Adam or SGD) to update the weights.
* **`imdb`**: Loads our data of movie reviews.

In [7]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tensorflow.keras.datasets import imdb

# Run on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

Using: cpu


### Step 2: Load and Prepare Data
* **Multi-Hot Vectorization**: Text reviews are stored as lists of word indices (e.g., `[1, 14, 22]`). We convert them into a 10,000-dimensional binary vector where `1.0` means the word is present, and `0.0` means it is absent. This format matches the input size of our network.

In [8]:
# 1. Load data (Limit to top 10k words)
num_words = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=num_words)

# 2. Vectorize function (Convert lists of integers to 1s and 0s)
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension), dtype=np.float32)
    for i, seq in enumerate(sequences):
        results[i, seq] = 1.0
    return results

X_train = vectorize_sequences(X_train, num_words)
X_test = vectorize_sequences(X_test, num_words)

# 3. Convert NumPy arrays directly to PyTorch Tensors
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

### Step 3: Define the Network Structure
* **`nn.Linear(input, output)`**: Connected layer (Dense layer).
* **`nn.ReLU()`**: Activation function that introduces non-linearity.
* **`nn.Dropout(p)`**: Randomly shuts down neurons during training to prevent memorization (overfitting).
* **`nn.Sigmoid()`**: Converts the final output into a probability between `0` (negative) and `1` (positive).

In [9]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim):
        super(SimpleClassifier, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 16), # Input Layer -> Hidden Layer 1
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(16, 16),       # Hidden Layer 1 -> Hidden Layer 2
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(16, 1),        # Hidden Layer 2 -> Output Layer
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.layers(x)

model = SimpleClassifier(num_words).to(device)
# BCELoss is standard for binary classification
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Step 4: Training Loop
This is the step-by-step training process:
1. **`optimizer.zero_grad()`**: Reset previous gradients so they don't accumulate.
2. **`model(inputs)`**: Feed-forward pass (make predictions).
3. **`criterion(predictions, targets)`**: Compute the difference (loss) between predictions and actual labels.
4. **`loss.backward()`**: Backpropagation (calculate gradients).
5. **`optimizer.step()`**: Update model weights using the optimizer.

In [10]:
epochs = 5 # Reduced for speed and simplicity during testing
batch_size = 512

model.train()
for epoch in range(epochs):
    # Mini-batch training loop
    for i in range(0, len(X_train_t), batch_size):
        # Slice mini-batches
        inputs = X_train_t[i:i+batch_size].to(device)
        targets = y_train_t[i:i+batch_size].to(device)

        # 1. Clear gradients
        optimizer.zero_grad()
        # 2. Forward pass
        outputs = model(inputs)
        # 3. Calculate Loss
        loss = criterion(outputs, targets)
        # 4. Backward pass (backpropagation)
        loss.backward()
        # 5. Update weights
        optimizer.step()

    print(f"Epoch {epoch+1:02d} completed | Loss: {loss.item():.4f}")

Epoch 01 completed | Loss: 0.6043
Epoch 02 completed | Loss: 0.4396
Epoch 03 completed | Loss: 0.3085
Epoch 04 completed | Loss: 0.2844
Epoch 05 completed | Loss: 0.2326


### Step 5: Test Evaluation
To measure final accuracy on unseen test reviews, we turn off Dropout using `model.eval()` and disable gradient updates using `torch.no_grad()`.

In [11]:
model.eval()
correct = 0
total = len(X_test_t)

with torch.no_grad():
    # Process test data in batches to fit memory
    for i in range(0, total, batch_size):
        inputs = X_test_t[i:i+batch_size].to(device)
        targets = y_test_t[i:i+batch_size].to(device)

        outputs = model(inputs)
        preds = (outputs > 0.5).float() # Threshold probability at 0.5
        correct += (preds == targets).sum().item()

accuracy = correct / total
print(f"Final Test Accuracy: {accuracy * 100:.2f}%")

Final Test Accuracy: 88.84%
